# 02. XGBoost Modeling

이 노트북은 10분/15분 데이터 각각에 대해 XGBoost 모델을 학습하고, Accuracy/F1/ROC-AUC와 피처 중요도를 확인합니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.train_xgboost import train_xgboost

DATASETS = [
    (PROJECT_ROOT / "data" / "Challenger_Ranked_Games_10minute.csv", "10minute"),
    (PROJECT_ROOT / "data" / "Challenger_Ranked_Games_15minute.csv", "15minute"),
]

## 빠른 실행

처음 실험할 때는 `quick=True`로 전체 코드가 정상 작동하는지 확인합니다.
최종 결과를 만들 때는 `quick=False`로 바꿔서 실행하면 됩니다.

In [ ]:
results = []
for data_file, label in DATASETS:
    metrics = train_xgboost(
        data_file=data_file,
        time_label=label,
        output_dir=PROJECT_ROOT / "results",
        model_dir=PROJECT_ROOT / "models",
        tune=True,
        quick=True,   # 최종 실험에서는 False 권장
    )
    results.append(metrics)

pd.DataFrame(results)

## 저장된 결과 확인

생성되는 주요 파일:
- `results/tables/xgboost_10minute_metrics.csv`
- `results/tables/xgboost_15minute_metrics.csv`
- `results/tables/xgboost_10minute_feature_importance.csv`
- `results/figures/xgboost_10minute_feature_importance.png`
- `results/figures/xgboost_10minute_confusion_matrix.png`
- `results/figures/xgboost_10minute_roc_curve.png`

In [ ]:
comparison = pd.concat([
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "xgboost_10minute_metrics.csv"),
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "xgboost_15minute_metrics.csv"),
], ignore_index=True)
comparison[["model", "time_label", "accuracy", "precision", "recall", "f1", "roc_auc"]]

In [ ]:
importance10 = pd.read_csv(PROJECT_ROOT / "results" / "tables" / "xgboost_10minute_feature_importance.csv")
importance15 = pd.read_csv(PROJECT_ROOT / "results" / "tables" / "xgboost_15minute_feature_importance.csv")

importance10.head(15), importance15.head(15)

## 발표용 해석 방향

XGBoost는 tabular data에서 강한 기본 성능을 보이는 모델입니다. 특히 피처 중요도를 통해 `골드 차이`, `레벨 차이`, `킬/어시스트 차이`, `오브젝트 차이` 중 어떤 요소가 승패 예측에 크게 작용했는지 설명할 수 있습니다.